In [21]:
import requests
import pandas as pd
import time
import random

In [23]:
target_subreddits = [
    'funny', 'pics', 'todayilearned', 'wholesomememes',
    'AskReddit', 'NoStupidQuestions', 'AmItheAsshole', 'tifu','worldnews',
    'technology', 'science',
    'gaming', 'movies', 'wallstreetbets', 'Music'
]

list_type = 'new'
list_type2 = 'hot'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36 OPR/105.0.0.0'
    }

num_pages = 20

In [24]:
for sub in target_subreddits:
  url = f"https://www.reddit.com/r/{sub}/{list_type}.json"
  after_tokens = None
  dataset = []

  for i in range(num_pages):
    params = {'limit':100}

    if after_tokens:
        params ['after'] = after_tokens

    attempt = 0
    response = None # Initialize response to None
    while attempt < 3:
      response = requests.get(url, headers=headers, params=params)

      if(response.status_code == 429):
        print(f"Rate limit hit for subreddit '{sub}' on page {i+1}. Retrying in 60 seconds...")
        time.sleep(60)
        attempt += 1
        continue
      elif response.status_code != 200:
        print (f"Failed to retrieve data for subreddit '{sub}' on page {i+1} with status code {response.status_code}. Breaking from retries.")
        break # Exit the while loop
      else: # response.status_code is 200
        break # Exit the while loop

    # Only proceed if the response was successful
    if response and response.status_code == 200:
        try:
            data = response.json()
            posts = data['data']['children']

            for post in posts:
              post_data = post['data']
              dataset.append({
                  'id': post_data['id'],
                  'subreddit': post_data['subreddit'],
                  'title': post_data['title'],
                  'score': post_data['score'],
                  'num_comments': post_data['num_comments'],
                  'created_utc': post_data['created_utc'],
                  'upvote_ratio': post_data['upvote_ratio'],
                  'is_self': post_data['is_self'],
                  'domain': post_data['domain']

              })

            after_tokens = data['data']['after']

            if not after_tokens:
              print(f"No more pages for subreddit '{sub}'.")
              break # Break out of the num_pages loop for the current subreddit

            time.sleep(random.uniform(1, 3))
        except requests.exceptions.JSONDecodeError as e:
            print(f"JSONDecodeError while parsing response for subreddit '{sub}' on page {i+1}: {e}. Skipping to next subreddit.")
            break # Break out of the num_pages loop for the current subreddit
    else:
        # If the request failed after retries or with a non-200 status code, skip further pages for this subreddit
        print(f"Skipping further pages for subreddit '{sub}' due to unsuccessful request.")
        break # Break out of the num_pages loop for the current subreddit

  if dataset:
    df = pd.DataFrame(dataset)
    df.to_csv(f'posts_{sub}_{list_type}.csv', index=False)
    print(f"Collected {len(df)} posts for subreddit '{sub}'.")
  else:
    print(f"No data collected for subreddit '{sub}'.")

  time.sleep(random.uniform(2, 5))

Failed to retrieve data for subreddit 'funny' on page 1 with status code 403. Breaking from retries.
Skipping further pages for subreddit 'funny' due to unsuccessful request.
No data collected for subreddit 'funny'.
Failed to retrieve data for subreddit 'pics' on page 1 with status code 403. Breaking from retries.
Skipping further pages for subreddit 'pics' due to unsuccessful request.
No data collected for subreddit 'pics'.
Failed to retrieve data for subreddit 'todayilearned' on page 1 with status code 403. Breaking from retries.
Skipping further pages for subreddit 'todayilearned' due to unsuccessful request.
No data collected for subreddit 'todayilearned'.
Failed to retrieve data for subreddit 'wholesomememes' on page 1 with status code 403. Breaking from retries.
Skipping further pages for subreddit 'wholesomememes' due to unsuccessful request.
No data collected for subreddit 'wholesomememes'.
Failed to retrieve data for subreddit 'AskReddit' on page 1 with status code 403. Breaki